# Three-Model Evaluation

Compare Transformer, LSTM, and FCN checkpoints on the same test groups and plot the shared ground truth once per group.

## Load libraries

In [ ]:
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (
    root_mean_squared_error,
    r2_score
)

from shaft_force_sensing.training.utils import load_model
from shaft_force_sensing.evaluation import tb_to_numpy, array_bais, array_medfilt

%load_ext autoreload
%autoreload 2

# Helpers

In [ ]:
METRICS_PATTERN = re.compile(
    r'^(?P<axis>[^:]+):\s*Range=(?P<Range>[-+0-9.eE]+),\s*'
    r'RMSE=(?P<RMSE>[-+0-9.eE]+),\s*'
    r'NRMSE=(?P<NRMSE>[-+0-9.eE]+)%,\s*'
    r'R2=(?P<R2>[-+0-9.eE]+)$'
)

In [ ]:
def parse_metrics_file(path: Path) -> tuple[dict, pd.DataFrame]:
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines()]

    if not lines or not lines[0].startswith('Model:'):
        raise ValueError(f'Unexpected metrics format: {path}')

    header = {'Model': lines[0].split(':', 1)[1].strip()}
    header['Test data'] = lines[1].split(':', 1)[1].strip()

    dashed = [i for i, ln in enumerate(lines) if ln == '-' * 10]
    if len(dashed) < 2:
        raise ValueError(f'Cannot locate hparams section in: {path}')

    hparam_lines = lines[dashed[0] + 1 : dashed[1]]
    for ln in hparam_lines:
        if ': ' in ln:
            k, v = ln.split(': ', 1)
            header[k] = v

    rows = []
    current_group = None
    for ln in lines[dashed[1] + 1 :]:
        if not ln or ln == '-' * 10:
            continue
        if ln.startswith('Group: '):
            current_group = ln.split(': ', 1)[1]
            continue

        m = METRICS_PATTERN.match(ln)
        if m and current_group is not None:
            row = {'group': current_group, 'axis': m.group('axis')}
            row.update({
                'Range': float(m.group('Range')),
                'RMSE': float(m.group('RMSE')),
                'NRMSE': float(m.group('NRMSE')),
                'R2': float(m.group('R2')),
            })
            rows.append(row)

    return header, pd.DataFrame(rows)

## Configure checkpoints

In [ ]:
LOG_ROOT = Path("../logs")

GROUP_DIRS = {
    "All inputs": LOG_ROOT / "ablations" / "full",
   r"No $F_x, F_y, F_z$": LOG_ROOT / "ablations" / "no_hex10",
   r"No $\tau$": LOG_ROOT / "ablations" / "no_tau",
   r"No $q$": LOG_ROOT / "ablations" / "no_pos",
   r"No $\dot{q}$": LOG_ROOT / "ablations" / "no_vec",
   r"No $q$ or $\dot{q}$": LOG_ROOT / "ablations" / "no_pos_vec",
}

In [ ]:
eval_dfs = {name: [] for name in GROUP_DIRS.keys()}

for name, dir in GROUP_DIRS.items():
    metrics_files = dir.parent.rglob(f"{dir.name}*/*/metrics.txt")
    for path in tqdm(metrics_files, desc=f"Parsing {name}"):
        _, df = parse_metrics_file(path)
        eval_dfs[name].append(df)
    eval_dfs[name] = pd.concat(eval_dfs[name], ignore_index=True)

In [ ]:
RMSEs = {name: df[df['group'] == 'All'].set_index('axis')['RMSE'] for name, df in eval_dfs.items()}
NRMSEs = {name: df[df['group'] == 'All'].set_index('axis')['NRMSE'] for name, df in eval_dfs.items()}
R2s = {name: df[df['group'] == 'All'].set_index('axis')['R2'] for name, df in eval_dfs.items()}

In [ ]:
axes_to_plot = ['F_x', 'F_y', 'F_z']
ablation_names = list(NRMSEs.keys())
r2_colors = {'F_x': 'red', 'F_y': 'blue', 'F_z': 'green'}
nrmse_colors = {'F_x': '#f4a261', 'F_y': '#8ecae6', 'F_z': '#cdb4db'}

fig, axs = plt.subplots(1, len(axes_to_plot), figsize=(5 * len(axes_to_plot), 5), sharey=True)

if len(axes_to_plot) == 1:
    axs = [axs]

base_positions = np.arange(1, len(ablation_names) + 1)

for ax, axis_name in zip(axs, axes_to_plot):
    nrmse_data = []
    r2_data = []

    for ablation in ablation_names:
        nrmse_series = NRMSEs[ablation]
        r2_series = R2s[ablation]

        if axis_name in nrmse_series.index:
            nrmse_vals = nrmse_series.loc[axis_name]
            if isinstance(nrmse_vals, pd.Series):
                nrmse_vals = nrmse_vals.dropna().to_numpy(dtype=float)
            else:
                nrmse_vals = np.array([nrmse_vals], dtype=float)
        else:
            nrmse_vals = np.array([np.nan], dtype=float)

        if axis_name in r2_series.index:
            r2_vals = r2_series.loc[axis_name]
            if isinstance(r2_vals, pd.Series):
                r2_vals = r2_vals.dropna().to_numpy(dtype=float)
            else:
                r2_vals = np.array([r2_vals], dtype=float)
        else:
            r2_vals = np.array([np.nan], dtype=float)

        if nrmse_vals.size == 0:
            nrmse_vals = np.array([np.nan], dtype=float)
        if r2_vals.size == 0:
            r2_vals = np.array([np.nan], dtype=float)

        nrmse_data.append(nrmse_vals)
        r2_data.append(r2_vals)

    nrmse_color = nrmse_colors[axis_name]
    r2_color = r2_colors[axis_name]

    # Bottom half: NRMSE violin on bottom x-axis
    vp_nrmse = ax.violinplot(
        nrmse_data,
        positions=base_positions,
        widths=0.62,
        vert=False,
        side='low',
        showmeans=False,
        showmedians=True,
        showextrema=True,
    )
    for body in vp_nrmse['bodies']:
        body.set_facecolor(nrmse_color)
        body.set_edgecolor(nrmse_color)
        body.set_alpha(0.90)
        body.set_linewidth(1.1)
    if 'cmedians' in vp_nrmse:
        vp_nrmse['cmedians'].set_color('black')
        vp_nrmse['cmedians'].set_linewidth(1.6)
    if 'cbars' in vp_nrmse:
        vp_nrmse['cbars'].set_color(nrmse_color)
        vp_nrmse['cbars'].set_linewidth(1.0)
    if 'cmins' in vp_nrmse:
        vp_nrmse['cmins'].set_color(nrmse_color)
        vp_nrmse['cmins'].set_linewidth(1.0)
    if 'cmaxes' in vp_nrmse:
        vp_nrmse['cmaxes'].set_color(nrmse_color)
        vp_nrmse['cmaxes'].set_linewidth(1.0)

    # Top half: R2 violin on top x-axis
    ax_top = ax.twiny()
    vp_r2 = ax_top.violinplot(
        r2_data,
        positions=base_positions,
        widths=0.62,
        vert=False,
        side='high',
        showmeans=False,
        showmedians=True,
        showextrema=True,
    )
    for body in vp_r2['bodies']:
        body.set_facecolor(r2_color)
        body.set_edgecolor(r2_color)
        body.set_alpha(0.55)
        body.set_linewidth(1.2)
    if 'cmedians' in vp_r2:
        vp_r2['cmedians'].set_color('black')
        vp_r2['cmedians'].set_linewidth(1.6)
    if 'cbars' in vp_r2:
        vp_r2['cbars'].set_color(r2_color)
        vp_r2['cbars'].set_linewidth(1.0)
    if 'cmins' in vp_r2:
        vp_r2['cmins'].set_color(r2_color)
        vp_r2['cmins'].set_linewidth(1.0)
    if 'cmaxes' in vp_r2:
        vp_r2['cmaxes'].set_color(r2_color)
        vp_r2['cmaxes'].set_linewidth(1.0)

    # Compute x-limits: R2 -> nearest integers, NRMSE -> nearest 0.25 multiples
    nrmse_all = np.concatenate([v[np.isfinite(v)] for v in nrmse_data if np.isfinite(v).any()]) if any(np.isfinite(v).any() for v in nrmse_data) else np.array([0.0, 1.0])
    r2_all = np.concatenate([v[np.isfinite(v)] for v in r2_data if np.isfinite(v).any()]) if any(np.isfinite(v).any() for v in r2_data) else np.array([0.0, 1.0])

    nrmse_min = np.floor(np.min(nrmse_all) / 0.25) * 0.25
    nrmse_max = np.ceil(np.max(nrmse_all) / 0.25) * 0.25
    if np.isclose(nrmse_min, nrmse_max):
        nrmse_max = nrmse_min + 0.25

    r2_min = float(np.floor(np.min(r2_all)))
    r2_max = float(np.ceil(np.max(r2_all)))
    if np.isclose(r2_min, r2_max):
        r2_max = r2_min + 1.0

    ax.set_xlim(nrmse_min, nrmse_max)
    ax_top.set_xlim(r2_min, r2_max)

    # Show only x-limit labels
    ax.set_xticks([nrmse_min, nrmse_max])
    ax.set_xticklabels([f'{nrmse_min:.2f}', f'{nrmse_max:.2f}'])
    ax_top.set_xticks([r2_min, r2_max])
    ax_top.set_xticklabels([f'{int(r2_min)}', f'{int(r2_max)}'])

    ax.set_title(f'${axis_name}$', color='black')
    ax.set_xlabel('NRMSE (%)', color='black')
    ax.grid(axis='x', alpha=0.3)
    ax.set_yticks(base_positions)
    ax.tick_params(axis='x', colors='black')
    ax.tick_params(axis='y', colors='black')

    ax_top.set_xlabel('$R^2$', color='black')
    ax_top.tick_params(axis='x', colors='black')
    ax_top.tick_params(axis='y', left=False, labelleft=False, right=False, labelright=False)

    for spine in ax.spines.values():
        spine.set_color('black')
    for spine in ax_top.spines.values():
        spine.set_color('black')

axs[0].set_yticks(base_positions)
axs[0].set_yticklabels(ablation_names, color='black')
for ax in axs[1:]:
    ax.tick_params(axis='y', labelleft=False)

fig.supylabel('Ablation', color='black')
# fig.suptitle('NRMSE (Violin, lower) and R2 (Violin, upper) Across Ablations', color='black')
plt.tight_layout(rect=[0.03, 0.0, 1.0, 0.95])

out_dir = Path('../logs/results/plots')
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'ablation_nrmse_r2_violin.pdf', bbox_inches='tight')

plt.show()